# Nusantara Alpha IDX Model Experiments

Local research notebook for developing and comparing IDX prediction models. This notebook uses historical OHLCV data, triple-barrier labels, chronological backtests, and educational simulated metrics. It is not financial advice, trading instruction, or a guarantee of future results.

Default baseline: pooled logistic regression with up, neutral, and down triple-barrier classes.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

mpl_cache = PROJECT_ROOT / ".local" / "matplotlib"
mpl_cache.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_cache))
MLFLOW_TRACKING_URI = f"sqlite:///{PROJECT_ROOT / '.local' / 'mlflow_tracking.sqlite'}"

PROJECT_ROOT

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from ml.backtesting.realistic import fit_pooled_logistic_model, predict_scores
from ml.experiments.baseline import (
    ExperimentConfig,
    load_price_rows_from_sqlite,
    run_baseline_logistic_experiment,
)
from ml.experiments.mlflow_tracking import log_baseline_experiment_to_mlflow

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 30)

## 1. Load Local OHLCV Data

The default path uses the local SQLite database created by the app ingestion flow. Set `TICKERS` to a small list while iterating quickly, or leave it as `None` to evaluate every supported stock in the local database.

In [ ]:
DB_PATH = PROJECT_ROOT / ".local" / "nusantara_alpha.sqlite3"
TICKERS = None  # Example for faster iteration: ["BBCA", "TLKM", "BMRI"]
START_DATE = None
END_DATE = None

price_rows = load_price_rows_from_sqlite(DB_PATH, tickers=TICKERS, start_date=START_DATE, end_date=END_DATE)
prices = pd.DataFrame(price_rows)
prices["price_date"] = pd.to_datetime(prices["price_date"])

summary = {
    "rows": len(prices),
    "tickers": prices["ticker"].nunique(),
    "first_date": prices["price_date"].min().date().isoformat(),
    "latest_date": prices["price_date"].max().date().isoformat(),
}
summary

In [ ]:
ticker_coverage = (
    prices.groupby("ticker")
    .agg(rows=("price_date", "size"), first_date=("price_date", "min"), latest_date=("price_date", "max"))
    .sort_values(["latest_date", "rows"], ascending=[True, True])
)
ticker_coverage.head(20)

## 2. Run The Baseline Chronological Backtest

This reproduces the baseline experiment framework: triple-barrier labels, a chronological train/test split, and a pooled logistic regression model. The simulated economic metrics are long-only for upward signals. Neutral and down predictions are still evaluated as classes, but they are not treated as short-selling instructions.

In [ ]:
config = ExperimentConfig(
    transaction_cost=0.0025,
    train_ratio=0.70,
    atr_window=20,
    barrier_atr_multiple=1.0,
    vertical_barrier_sessions=5,
)

result = run_baseline_logistic_experiment(price_rows, config)
result.summary

## 3. Log The Experiment To MLflow

Run this cell when you want to keep the experiment. Promotion to the customer-facing app is a separate manual approval step; this only creates an MLflow experiment run and model artifact.

In [ ]:
logged = log_baseline_experiment_to_mlflow(
    result,
    price_rows,
    tracking_uri=MLFLOW_TRACKING_URI,
    experiment_name="nusantara-alpha-local-experiments",
    run_name="baseline-logistic-triple-barrier",
    candidate_model_id="idx-logistic-experiment",
    candidate_model_version="local-candidate",
)

logged

In [ ]:
metric_columns = [
    "ticker",
    "test_rows",
    "directional_accuracy",
    "upward_precision",
    "signal_coverage",
    "up_signal_count",
    "neutral_signal_count",
    "down_signal_count",
    "average_daily_simulated_return",
    "cumulative_return_after_cost",
    "max_drawdown",
    "sharpe_like",
]
result.per_stock[metric_columns].head(25)

## 4. Economic Metric Views

These charts show historical simulated behavior under the experiment assumptions. They are diagnostic research views, not recommendations.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
result.daily_returns.plot(x="price_date", y="equity_curve", ax=axes[0], legend=False, title="Aggregate Simulated Equity Curve")
axes[0].set_ylabel("Equity multiple")
result.daily_returns.plot(x="price_date", y="drawdown", ax=axes[1], legend=False, title="Aggregate Drawdown")
axes[1].set_ylabel("Drawdown")
axes[1].set_xlabel("Date")
plt.tight_layout()

In [ ]:
signal_counts = result.predictions["model_signal"].value_counts().reindex(["up", "neutral", "down"], fill_value=0)
signal_counts.plot(kind="bar", title="Held-Out Signal Distribution", figsize=(8, 4))
plt.ylabel("Rows")
plt.xticks(rotation=0)
plt.tight_layout()

In [ ]:
top_by_return = result.per_stock.sort_values("cumulative_return_after_cost", ascending=False).head(20)
bottom_by_return = result.per_stock.sort_values("cumulative_return_after_cost", ascending=True).head(20)

display(top_by_return[metric_columns])
display(bottom_by_return[metric_columns])

## 5. Latest Baseline Scores

This fits the baseline on the loaded data and scores the latest feature row for each ticker. Same data plus same model settings should produce the same ranking.

In [ ]:
latest_model = fit_pooled_logistic_model(
    price_rows,
    atr_window=config.atr_window,
    barrier_atr_multiple=config.barrier_atr_multiple,
    vertical_barrier_sessions=config.vertical_barrier_sessions,
)
latest_scores = predict_scores(latest_model)
latest_scores[["ticker", "price_date", "model_signal", "model_score", "up_probability", "neutral_probability", "down_probability", "rank"]].head(30)

## 6. Experiment Sandbox

Change one thing at a time. Useful first experiments: barrier width, vertical window, feature columns in `ml/backtesting/realistic.py`, or a new model family inside a separate helper.

In [ ]:
sandbox_config = ExperimentConfig(
    transaction_cost=0.0025,
    train_ratio=0.70,
    atr_window=20,
    barrier_atr_multiple=1.5,
    vertical_barrier_sessions=10,
)

sandbox_result = run_baseline_logistic_experiment(price_rows, sandbox_config)
pd.DataFrame([result.summary, sandbox_result.summary], index=["baseline", "sandbox"]).T